# 中证800 V86：V46 样本碰撞与特征可分辨性审计

本 notebook 不寻找新模型，而是判断 V46 的主要瓶颈究竟来自数据错误、特征覆盖不足，还是一个月个股收益本身的高条件噪声。

审计分三层：

1. **结构质量**：重复股票月、非有限标签、特征缺失、整行重复、特征停滞和极端标签。
2. **同月碰撞**：对每个 OOS 月份的 V46 特征做截面秩标准化，比较最近邻与同行业随机配对的标签差异。
3. **严格历史 OOS 近邻**：测试样本只能从训练截止日前且标签已经实现的历史样本中寻找近邻，检验特征几何是否包含可跨时间复用的关系。

同月碰撞是诊断而不是回测；历史近邻才是预测性证据。所有距离计算逐月或分批执行，避免构造全样本距离矩阵。


## 0. 导入、进度条与绘图基础


In [ ]:
import gc
import math
import warnings
import builtins as _bi
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 260)
pd.set_option("display.width", 280)
pd.set_option("display.max_rows", 160)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = _bi.max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=30):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


def save_show(fig, filename):
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## 1. 实验配置与预注册阈值


In [ ]:
PROJECT_DIR = Path.cwd()
OUT_DIR = PROJECT_DIR / "csi800_ml_v86_sample_collision_audit_outputs"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("train_csi800_factor_v40_data_enhancement.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement.csv"),
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement.csv",
]
DATA_PATH_OVERRIDE = None

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
INDUSTRY_COL = "industry_bucket"
RANK_PREFIX = "__rank__"

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
STOCK_NUM = 8
TRUE_TOP_N = 20
LOCAL_K = 5
TOP_CANDIDATE_N = 30
CASES_PER_MONTH = 12
GROUP_SAMPLE_PER_MONTH = 300
HIST_POOL_PER_MONTH = 160
HIST_QUERY_BATCH = 100
HIST_K_LIST = [5, 20]
HISTORICAL_KNN_LABEL_SAFE = True

# Pre-registered interpretation thresholds.
GOOD_GAP_REDUCTION = 0.25
MIXED_GAP_REDUCTION = 0.10
GOOD_SIGN_CONFLICT_MAX = 0.35
BAD_SIGN_CONFLICT_MIN = 0.45
GOOD_LOCAL_STD_RATIO_MAX = 0.75
BAD_LOCAL_STD_RATIO_MIN = 0.90
GOOD_HIST_KNN_RANK_IC = 0.03
MIXED_HIST_KNN_RANK_IC = 0.015

FOLD_PLAN = [
    {"fold_id": "cutoff202112", "train_start": "2019-01-01", "train_end": "2021-12-31", "test_start": "2022-01-01", "test_end": "2022-12-31"},
    {"fold_id": "cutoff202212", "train_start": "2019-01-01", "train_end": "2022-12-31", "test_start": "2023-01-01", "test_end": "2023-12-31"},
    {"fold_id": "cutoff202312", "train_start": "2019-01-01", "train_end": "2023-12-31", "test_start": "2024-01-01", "test_end": "2024-12-31"},
    {"fold_id": "cutoff202412", "train_start": "2019-01-01", "train_end": "2024-12-31", "test_start": "2025-01-01", "test_end": "2025-12-31"},
    {"fold_id": "cutoff202512", "train_start": "2019-01-01", "train_end": "2025-12-31", "test_start": "2026-01-01", "test_end": "2026-12-31"},
]

SMOKE_TEST = False
if SMOKE_TEST:
    FOLD_PLAN = FOLD_PLAN[:1]
    GROUP_SAMPLE_PER_MONTH = 150
    HIST_POOL_PER_MONTH = 60
    CASES_PER_MONTH = 5

COLORS = {
    "selected_equal": "#2f5597",
    "selected_gain": "#00a087",
    "model_score": "#e64b35",
    "industry_random": "#7f7f7f",
    "hist_knn5": "#7e57c2",
    "hist_knn20": "#f0a202",
    "v46_score": "#2f5597",
}

print("OUT_DIR:", OUT_DIR)
print("FOLDS:", [x["fold_id"] for x in FOLD_PLAN])


## 2. V46 特征、因子组与固定模型参数


In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]
HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m",
]
FULL_FEATURE_COLS = unique_keep_order(BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS)

FACTOR_GROUPS = {
    "valuation": [
        "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
        "cash_earnings_to_price_ratio", "earnings_to_price_ratio",
    ],
    "profitability_quality": [
        "roe_ttm", "roa_ttm", "gross_profit_ttm", "operating_profit_to_total_profit",
        "net_operate_cash_flow_to_total_liability", "net_operating_cash_flow_coverage",
        "adjusted_profit_to_total_profit", "ACCA", "growth",
    ],
    "balance_sheet_per_share": [
        "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
        "total_operating_revenue_per_share", "super_quick_ratio", "MLEV",
        "debt_to_equity_ratio", "debt_to_tangible_equity_ratio",
    ],
    "momentum_risk": [
        "momentum", "Rank1M", "sharpe_ratio_60", "Variance20", "beta", "ATR6",
        "Skewness20", "Kurtosis20",
    ],
    "liquidity_volume": [
        "liquidity", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
        "liq_money_ratio_20_60", "liq_paused_count_20",
    ],
    "price_time_series": [
        "px_close_to_ma60", "px_drawdown_60",
        "ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m",
    ],
}

BASE_PARAMS_FF10 = {
    "objective": "regression", "metric": "l2", "boosting_type": "gbdt",
    "learning_rate": 0.05, "num_leaves": 31, "min_data_in_leaf": 200,
    "feature_fraction": 1.0, "bagging_fraction": 0.8, "bagging_freq": 1,
    "lambda_l1": 0.1, "lambda_l2": 0.3, "verbose": -1,
}

group_manifest_rows = []
for group_name, cols in FACTOR_GROUPS.items():
    for col in cols:
        group_manifest_rows.append({"factor_group": group_name, "feature": col})
factor_group_manifest_df = pd.DataFrame(group_manifest_rows)
factor_group_manifest_df.to_csv(OUT_DIR / "v86_factor_group_manifest.csv", index=False)
display_df(factor_group_manifest_df, 60)


## 3. 数据、训练与统计工具


In [ ]:
def resolve_data_path():
    if DATA_PATH_OVERRIDE:
        p = Path(DATA_PATH_OVERRIDE)
        if p.exists():
            return p
        raise IOError("DATA_PATH_OVERRIDE not found: %s" % p)
    for raw in DATA_CANDIDATES:
        p = Path(raw)
        if p.exists():
            return p
    raise IOError("training data csv not found: %s" % [str(Path(x).resolve()) for x in DATA_CANDIDATES])


def safe_rank_ic(a, b):
    d = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < 3 or d["a"].nunique() < 2 or d["b"].nunique() < 2:
        return np.nan
    return float(d["a"].rank(method="average").corr(d["b"].rank(method="average")))


def load_dataset(path):
    header = pd.read_csv(path, nrows=0)
    available = set(header.columns)
    stock_col = STOCK_COL
    if stock_col not in available:
        for alt in ["code", "security", "order_book_id"]:
            if alt in available:
                stock_col = alt
                break
    required = [stock_col, DATE_COL, TARGET_COL] + FULL_FEATURE_COLS
    optional = [INDUSTRY_COL, "feature_date", "next_date", "raw_return_1m", "benchmark_csi800_1m"]
    missing = [c for c in required if c not in available]
    if missing:
        raise ValueError("dataset missing columns: " + ",".join(missing))
    usecols = unique_keep_order(required + [c for c in optional if c in available])
    df = pd.read_csv(path, usecols=usecols)
    if stock_col != STOCK_COL:
        df = df.rename(columns={stock_col: STOCK_COL})
    for col in [DATE_COL, "feature_date", "next_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce").dt.normalize()
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    if "feature_date" not in df.columns:
        df["feature_date"] = df[DATE_COL]
    if "next_date" not in df.columns:
        df["next_date"] = df[DATE_COL]
    df[STOCK_COL] = df[STOCK_COL].astype(str)
    numeric_cols = FULL_FEATURE_COLS + [TARGET_COL]
    for col in progress_iter(numeric_cols, total=len(numeric_cols), desc="compact numeric float32"):
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).astype(np.float32)
    df = df.dropna(subset=[STOCK_COL, DATE_COL, TARGET_COL]).copy()
    df = df.sort_values([DATE_COL, STOCK_COL]).reset_index(drop=True)
    stock_values = _bi.sorted(df[STOCK_COL].unique().tolist())
    stock_to_id = dict((s, i) for i, s in enumerate(stock_values))
    df["__stock_id"] = df[STOCK_COL].map(stock_to_id).astype(np.int32)
    return df


def make_train_df(df, fold, label_safe=False):
    start = pd.Timestamp(fold["train_start"])
    end = pd.Timestamp(fold["train_end"])
    mask = (df[DATE_COL] >= start) & (df[DATE_COL] <= end)
    if label_safe:
        mask = mask & (df["next_date"] <= end)
    return df.loc[mask].copy()


def make_test_df(df, fold):
    start = pd.Timestamp(fold["test_start"])
    end = pd.Timestamp(fold["test_end"])
    return df.loc[(df[DATE_COL] >= start) & (df[DATE_COL] <= end)].copy()


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            value = corr.iloc[i, j]
            if not pd.isnull(value) and abs(value) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    components = []
    def dfs(node, component):
        visited.add(node)
        component.append(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                dfs(neighbor, component)
    for col in feature_cols:
        if col not in visited:
            component = []
            dfs(col, component)
            components.append(component)
    return components


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for component in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(component) == 1:
            keep.append(component[0])
        else:
            ordered = _bi.sorted(component, key=lambda x: (missing[x], x))
            keep.append(ordered[0])
            remove.extend(ordered[1:])
    if len(keep) == 0:
        raise ValueError("no usable V46 features")
    return keep, remove


def prepare_x(df, feature_cols, fill_values=None):
    x = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    if fill_values is None:
        fill_values = x.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    return x.fillna(fill_values).fillna(0), fill_values


def train_v46(train_df, feature_cols):
    work = train_df.sort_values([DATE_COL, STOCK_COL]).dropna(subset=[TARGET_COL]).copy()
    x, fill_values = prepare_x(work, feature_cols, None)
    y = pd.to_numeric(work[TARGET_COL], errors="coerce")
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    dataset = lgb.Dataset(x[feature_cols], label=np.asarray(y))
    model = lgb.train(params, dataset, num_boost_round=_bi.max(1, int(FIXED_ITER)))
    pred = np.asarray(model.predict(x[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    return {
        "model": model, "fill_values": fill_values, "train_rows": int(len(work)),
        "train_rank_ic": safe_rank_ic(pred, y),
    }


def score_v46(df, trained, feature_cols):
    x, _ = prepare_x(df, feature_cols, trained["fill_values"])
    return np.asarray(trained["model"].predict(x[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)


def add_monthly_rank_features(df):
    rank_cols = [RANK_PREFIX + c for c in FULL_FEATURE_COLS]
    rank_values = np.full((len(df), len(FULL_FEATURE_COLS)), np.nan, dtype=np.float32)
    groups = list(df.groupby(DATE_COL).groups.items())
    for dt, index_values in progress_iter(groups, total=len(groups), desc="monthly feature rank transform"):
        idx = np.asarray(index_values, dtype=int)
        for j, col in enumerate(FULL_FEATURE_COLS):
            values = pd.to_numeric(df.loc[idx, col], errors="coerce")
            valid = values.notnull()
            n_valid = int(valid.sum())
            if n_valid:
                ranks = values.loc[valid].rank(method="average") / float(n_valid + 1)
                rank_values[np.asarray(ranks.index, dtype=int), j] = np.asarray(ranks, dtype=np.float32)
    rank_df = pd.DataFrame(rank_values, columns=rank_cols)
    return pd.concat([df.reset_index(drop=True), rank_df], axis=1), rank_cols


## 4. 近邻距离、碰撞指标与历史 OOS KNN 工具


In [ ]:
def fill_rank_matrix(df, rank_cols):
    x = np.asarray(df[rank_cols], dtype=np.float32).copy()
    x[~np.isfinite(x)] = 0.5
    return x


def pairwise_rank_distance(x, weights=None):
    x = np.asarray(x, dtype=np.float32)
    if weights is None:
        weights = np.ones(x.shape[1], dtype=np.float32) / float(_bi.max(1, x.shape[1]))
    else:
        weights = np.asarray(weights, dtype=np.float32)
        weights = weights / float(weights.sum()) if float(weights.sum()) > 0 else np.ones(x.shape[1], dtype=np.float32) / float(x.shape[1])
    z = x * np.sqrt(weights.reshape(1, -1))
    sq = np.square(z).sum(axis=1)
    dist2 = sq.reshape(-1, 1) + sq.reshape(1, -1) - 2.0 * np.dot(z, z.T)
    dist2[dist2 < 0] = 0
    np.fill_diagonal(dist2, np.inf)
    return np.sqrt(dist2)


def random_industry_partner(industry_values, rng):
    industry_values = np.asarray(industry_values).astype(str)
    n = len(industry_values)
    result = np.zeros(n, dtype=int)
    all_idx = np.arange(n)
    by_industry = {}
    for industry in set(industry_values.tolist()):
        by_industry[industry] = np.where(industry_values == industry)[0]
    for i in range(n):
        candidates = by_industry.get(industry_values[i], all_idx)
        candidates = candidates[candidates != i]
        if len(candidates) == 0:
            candidates = all_idx[all_idx != i]
        result[i] = int(rng.choice(candidates))
    return result


def pair_metrics(labels, partner_idx, distances, true_rank, local_neighbor_idx=None):
    labels = np.asarray(labels, dtype=float)
    partner_labels = labels[np.asarray(partner_idx, dtype=int)]
    gaps = np.abs(labels - partner_labels)
    sign_conflict = (labels > 0) != (partner_labels > 0)
    tail_n = int(_bi.min(TRUE_TOP_N, _bi.max(1, len(labels) // 4)))
    top20 = true_rank <= tail_n
    bottom20 = true_rank > (len(labels) - tail_n)
    extreme_collision = (top20 & bottom20[np.asarray(partner_idx, dtype=int)]) | (bottom20 & top20[np.asarray(partner_idx, dtype=int)])
    result = {
        "pairs": int(len(labels)),
        "distance_mean": float(np.nanmean(distances)),
        "distance_median": float(np.nanmedian(distances)),
        "abs_label_gap_mean": float(np.nanmean(gaps)),
        "abs_label_gap_median": float(np.nanmedian(gaps)),
        "sign_conflict_rate": float(np.mean(sign_conflict)),
        "label_gap_gt10_rate": float(np.mean(gaps > 0.10)),
        "label_gap_gt20_rate": float(np.mean(gaps > 0.20)),
        "top20_bottom20_collision_rate": float(np.mean(extreme_collision)),
    }
    if local_neighbor_idx is not None:
        local_labels = labels[np.asarray(local_neighbor_idx, dtype=int)]
        month_std = float(np.nanstd(labels, ddof=1))
        local_std = np.nanstd(local_labels, axis=1, ddof=1)
        result["local_label_std_mean"] = float(np.nanmean(local_std))
        result["local_std_ratio"] = float(np.nanmean(local_std) / month_std) if month_std > 0 else np.nan
        result["same_month_knn_rank_ic"] = safe_rank_ic(np.nanmean(local_labels, axis=1), labels)
    else:
        result["local_label_std_mean"] = np.nan
        result["local_std_ratio"] = np.nan
        result["same_month_knn_rank_ic"] = np.nan
    return result, gaps


def analyze_distance_matrix(month_df, dist, method, fold_id, rng, case_limit=CASES_PER_MONTH):
    labels = np.asarray(month_df[TARGET_COL], dtype=float)
    n = len(month_df)
    true_rank = np.asarray(pd.Series(labels).rank(method="first", ascending=False), dtype=float)
    nearest_idx = np.argmin(dist, axis=1)
    nearest_distance = dist[np.arange(n), nearest_idx]
    k = int(_bi.min(LOCAL_K, n - 1))
    local_idx = np.argpartition(dist, kth=k - 1, axis=1)[:, :k]
    result, gaps = pair_metrics(labels, nearest_idx, nearest_distance, true_rank, local_idx)
    result.update({"fold_id": fold_id, DATE_COL: pd.Timestamp(month_df[DATE_COL].iloc[0]), "method": method})
    mutual = np.asarray([nearest_idx[nearest_idx[i]] == i for i in range(n)], dtype=bool)
    result["mutual_nearest_rate"] = float(mutual.mean())

    close_cut = float(np.nanpercentile(nearest_distance, 20))
    case_mask = nearest_distance <= close_cut
    case_idx = np.where(case_mask)[0]
    case_idx = case_idx[np.argsort(gaps[case_idx])[::-1][:_bi.min(case_limit, len(case_idx))]]
    case_rows = []
    stocks = month_df[STOCK_COL].astype(str).values
    industries = month_df[INDUSTRY_COL].astype(str).values
    scores = np.asarray(month_df["v46_score"], dtype=float)
    for i in case_idx:
        j = int(nearest_idx[i])
        case_rows.append({
            "fold_id": fold_id, DATE_COL: pd.Timestamp(month_df[DATE_COL].iloc[0]), "method": method,
            "stock": stocks[i], "neighbor_stock": stocks[j], "industry": industries[i],
            "neighbor_industry": industries[j], "distance": float(nearest_distance[i]),
            "alpha": float(labels[i]), "neighbor_alpha": float(labels[j]),
            "abs_label_gap": float(gaps[i]), "stock_true_rank": int(true_rank[i]),
            "neighbor_true_rank": int(true_rank[j]), "v46_score": float(scores[i]),
            "neighbor_v46_score": float(scores[j]), "mutual_nearest": bool(mutual[i]),
        })
    return result, case_rows, nearest_idx


def score_distance_matrix(scores):
    values = np.asarray(scores, dtype=np.float32)
    scale = float(np.nanstd(values))
    scale = scale if scale > 1e-12 else 1.0
    dist = np.abs(values.reshape(-1, 1) - values.reshape(1, -1)) / scale
    np.fill_diagonal(dist, np.inf)
    return dist


def top8_metrics(labels, scores):
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)
    n = len(labels)
    order = np.argsort(scores)[::-1]
    true_order = np.argsort(labels)[::-1]
    selected = order[:_bi.min(STOCK_NUM, n)]
    true20 = set(true_order[:_bi.min(TRUE_TOP_N, n)].tolist())
    universe_mean = float(np.nanmean(labels))
    return {
        "rank_ic": safe_rank_ic(scores, labels),
        "precision_at8_true20": float(len(set(selected.tolist()) & true20)) / float(_bi.max(1, len(selected))),
        "recall_at8_true20": float(len(set(selected.tolist()) & true20)) / float(TRUE_TOP_N),
        "top8_alpha": float(np.nanmean(labels[selected])),
        "top8_edge": float(np.nanmean(labels[selected]) - universe_mean),
        "top_decile_alpha": float(np.nanmean(labels[order[:_bi.max(1, int(math.ceil(n * 0.10)))]])),
    }


def sample_history_pool(train_df, per_month, seed):
    rng = np.random.RandomState(seed)
    pieces = []
    groups = list(train_df.groupby(DATE_COL))
    for dt, gdf in groups:
        if len(gdf) > per_month:
            idx = rng.choice(np.arange(len(gdf)), size=int(per_month), replace=False)
            pieces.append(gdf.iloc[idx])
        else:
            pieces.append(gdf)
    return pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()


def historical_knn_predictions(query_df, pool_df, rank_cols, k_list, weights=None):
    pool_x = fill_rank_matrix(pool_df, rank_cols)
    pool_y = np.asarray(pool_df[TARGET_COL], dtype=np.float32)
    pool_stock = np.asarray(pool_df["__stock_id"], dtype=np.int32)
    if weights is None:
        weights = np.ones(len(rank_cols), dtype=np.float32) / float(len(rank_cols))
    weights = np.asarray(weights, dtype=np.float32)
    weights = weights / float(weights.sum())
    pool_z = pool_x * np.sqrt(weights.reshape(1, -1))
    pool_sq = np.square(pool_z).sum(axis=1)
    max_k = int(_bi.max(k_list))
    outputs = dict((k, np.full(len(query_df), np.nan, dtype=np.float32)) for k in k_list)
    mean_distance = np.full(len(query_df), np.nan, dtype=np.float32)
    for start in progress_iter(range(0, len(query_df), HIST_QUERY_BATCH), total=int(math.ceil(len(query_df) / float(HIST_QUERY_BATCH))), desc="historical KNN batches", leave=False):
        end = _bi.min(start + HIST_QUERY_BATCH, len(query_df))
        q = fill_rank_matrix(query_df.iloc[start:end], rank_cols)
        qz = q * np.sqrt(weights.reshape(1, -1))
        qsq = np.square(qz).sum(axis=1)
        dist2 = qsq.reshape(-1, 1) + pool_sq.reshape(1, -1) - 2.0 * np.dot(qz, pool_z.T)
        dist2[dist2 < 0] = 0
        qstock = np.asarray(query_df.iloc[start:end]["__stock_id"], dtype=np.int32)
        dist2[qstock.reshape(-1, 1) == pool_stock.reshape(1, -1)] = np.inf
        idx = np.argpartition(dist2, kth=max_k - 1, axis=1)[:, :max_k]
        batch_rows = np.arange(len(idx)).reshape(-1, 1)
        selected_dist = dist2[batch_rows, idx]
        order = np.argsort(selected_dist, axis=1)
        row_idx = np.arange(len(idx)).reshape(-1, 1)
        idx = idx[row_idx, order]
        selected_dist = selected_dist[row_idx, order]
        for k in k_list:
            outputs[k][start:end] = np.nanmean(pool_y[idx[:, :int(k)]], axis=1)
        mean_distance[start:end] = np.sqrt(np.nanmean(selected_dist[:, :_bi.min(LOCAL_K, max_k)], axis=1))
        del q, qz, dist2, idx, selected_dist
        gc.collect()
    return outputs, mean_distance


## 5. 加载数据并审计结构质量与标签尾部


In [ ]:
DATA_PATH = resolve_data_path()
df_all = load_dataset(DATA_PATH)

duplicate_stock_date = int(df_all.duplicated([STOCK_COL, DATE_COL]).sum())
nonfinite_target = int((~np.isfinite(np.asarray(df_all[TARGET_COL], dtype=float))).sum())

feature_quality_rows = []
same_fraction_count = np.zeros(len(df_all), dtype=np.int16)
comparable_count = np.zeros(len(df_all), dtype=np.int16)
for col in progress_iter(FULL_FEATURE_COLS, total=len(FULL_FEATURE_COLS), desc="feature quality and staleness"):
    values = pd.to_numeric(df_all[col], errors="coerce")
    previous = df_all.groupby(STOCK_COL)[col].shift(1)
    comparable = values.notnull() & previous.notnull()
    tolerance = 1e-10 * (1.0 + previous.abs())
    same = comparable & ((values - previous).abs() <= tolerance)
    same_fraction_count += np.asarray(same, dtype=np.int16)
    comparable_count += np.asarray(comparable, dtype=np.int16)
    zero_variance_months = 0
    for dt, month_values in df_all.groupby(DATE_COL)[col]:
        valid = pd.to_numeric(month_values, errors="coerce").dropna()
        if len(valid) == 0 or valid.nunique() <= 1:
            zero_variance_months += 1
    feature_quality_rows.append({
        "feature": col, "rows": int(len(values)), "missing_count": int(values.isnull().sum()),
        "missing_rate": float(values.isnull().mean()), "zero_variance_months": int(zero_variance_months),
        "consecutive_comparable_rows": int(comparable.sum()),
        "consecutive_same_value_rate": float(same.sum()) / float(_bi.max(1, int(comparable.sum()))),
        "p01": float(values.quantile(0.01)), "median": float(values.median()), "p99": float(values.quantile(0.99)),
    })
feature_quality_df = pd.DataFrame(feature_quality_rows)
stale_fraction = np.divide(
    same_fraction_count.astype(float), comparable_count.astype(float),
    out=np.full(len(df_all), np.nan, dtype=float), where=comparable_count > 0,
)
df_all["__consecutive_same_feature_fraction"] = stale_fraction.astype(np.float32)

feature_hash = pd.util.hash_pandas_object(df_all[FULL_FEATURE_COLS], index=False)
hash_frame = pd.DataFrame({DATE_COL: df_all[DATE_COL].values, "feature_hash": np.asarray(feature_hash).astype(str)})
exact_duplicate_mask = hash_frame.duplicated([DATE_COL, "feature_hash"], keep=False)
exact_duplicate_cases_df = df_all.loc[exact_duplicate_mask, [DATE_COL, STOCK_COL, INDUSTRY_COL, TARGET_COL]].copy()
exact_duplicate_cases_df["feature_hash"] = hash_frame.loc[exact_duplicate_mask, "feature_hash"].values
exact_duplicate_conflict_groups = 0
if len(exact_duplicate_cases_df):
    for _, gdf in exact_duplicate_cases_df.groupby([DATE_COL, "feature_hash"]):
        if float(pd.to_numeric(gdf[TARGET_COL], errors="coerce").max() - pd.to_numeric(gdf[TARGET_COL], errors="coerce").min()) > 0.10:
            exact_duplicate_conflict_groups += 1

label_monthly_rows = []
for dt, gdf in progress_iter(df_all.groupby(DATE_COL), total=df_all[DATE_COL].nunique(), desc="label tail audit"):
    y = pd.to_numeric(gdf[TARGET_COL], errors="coerce").dropna()
    if len(y) == 0:
        continue
    centered = np.asarray(y - y.mean(), dtype=float)
    loss = np.square(centered)
    tail_n = int(_bi.max(1, math.ceil(len(y) * 0.05)))
    tail_idx = np.argsort(np.abs(centered))[::-1][:tail_n]
    label_monthly_rows.append({
        DATE_COL: pd.Timestamp(dt), "n": int(len(y)), "mean": float(y.mean()), "std": float(y.std()),
        "p01": float(y.quantile(0.01)), "p05": float(y.quantile(0.05)), "median": float(y.median()),
        "p95": float(y.quantile(0.95)), "p99": float(y.quantile(0.99)),
        "abs_gt20_rate": float((y.abs() > 0.20).mean()), "abs_gt50_rate": float((y.abs() > 0.50).mean()),
        "top5pct_centered_l2_share": float(loss[tail_idx].sum() / loss.sum()) if loss.sum() > 0 else np.nan,
    })
label_monthly_df = pd.DataFrame(label_monthly_rows)
label_monthly_df["year"] = label_monthly_df[DATE_COL].dt.year

extreme_cols = [DATE_COL, "feature_date", "next_date", STOCK_COL, INDUSTRY_COL, TARGET_COL]
extreme_cols = [c for c in extreme_cols if c in df_all.columns]
extreme_index = df_all[TARGET_COL].abs().sort_values(ascending=False).head(200).index
extreme_label_cases_df = df_all.loc[extreme_index, extreme_cols].copy()
extreme_label_cases_df["abs_alpha"] = extreme_label_cases_df[TARGET_COL].abs()
extreme_label_cases_df = extreme_label_cases_df.sort_values("abs_alpha", ascending=False)

stale_rows = df_all[df_all["__consecutive_same_feature_fraction"] >= 0.90].copy()
stale_summary_df = pd.DataFrame([{
    "rows_with_previous_features": int(np.isfinite(stale_fraction).sum()),
    "rows_same_feature_fraction_ge_90pct": int(len(stale_rows)),
    "rate_same_feature_fraction_ge_90pct": float(len(stale_rows)) / float(_bi.max(1, int(np.isfinite(stale_fraction).sum()))),
    "stale_rows_label_abs_mean": float(stale_rows[TARGET_COL].abs().mean()) if len(stale_rows) else np.nan,
    "all_rows_label_abs_mean": float(df_all[TARGET_COL].abs().mean()),
}])

structural_audit_df = pd.DataFrame([
    {"check": "rows", "value": int(len(df_all))},
    {"check": "months", "value": int(df_all[DATE_COL].nunique())},
    {"check": "date_min", "value": str(df_all[DATE_COL].min())},
    {"check": "date_max", "value": str(df_all[DATE_COL].max())},
    {"check": "duplicate_stock_date", "value": duplicate_stock_date},
    {"check": "nonfinite_target", "value": nonfinite_target},
    {"check": "exact_duplicate_feature_rows", "value": int(exact_duplicate_mask.sum())},
    {"check": "exact_duplicate_conflict_groups_gap_gt10pct", "value": int(exact_duplicate_conflict_groups)},
    {"check": "feature_count", "value": int(len(FULL_FEATURE_COLS))},
    {"check": "lightgbm_version", "value": str(getattr(lgb, "__version__", "unknown"))},
])

structural_audit_df.to_csv(OUT_DIR / "v86_structural_audit.csv", index=False)
feature_quality_df.to_csv(OUT_DIR / "v86_feature_quality.csv", index=False)
stale_summary_df.to_csv(OUT_DIR / "v86_stale_feature_summary.csv", index=False)
exact_duplicate_cases_df.to_csv(OUT_DIR / "v86_exact_duplicate_feature_cases.csv", index=False)
label_monthly_df.to_csv(OUT_DIR / "v86_label_monthly_audit.csv", index=False)
extreme_label_cases_df.to_csv(OUT_DIR / "v86_extreme_label_cases.csv", index=False)

print("DATA_PATH:", DATA_PATH)
print("loaded:", df_all.shape)
display_df(structural_audit_df, 30)
display_df(feature_quality_df.sort_values("missing_rate", ascending=False), 50)
display_df(stale_summary_df, 10)
display_df(label_monthly_df.describe().T, 30)

df_all, ALL_RANK_COLS = add_monthly_rank_features(df_all)
print("rank-enriched shape:", df_all.shape)


## 6. Walk-forward V46、同月碰撞与历史 OOS 近邻


In [ ]:
collision_monthly_rows = []
collision_case_rows = []
top30_collision_rows = []
group_collision_rows = []
historical_knn_rows = []
model_meta_rows = []
feature_importance_rows = []

for fold_index, fold in enumerate(progress_iter(FOLD_PLAN, total=len(FOLD_PLAN), desc="collision audit folds")):
    train_df = make_train_df(df_all, fold, label_safe=False)
    test_df = make_test_df(df_all, fold)
    if train_df.empty or test_df.empty:
        print("skip empty fold", fold["fold_id"])
        del train_df, test_df
        gc.collect()
        continue

    selected_features, removed_features = select_features_train_only(train_df, FULL_FEATURE_COLS)
    selected_rank_cols = [RANK_PREFIX + c for c in selected_features]
    trained = train_v46(train_df, selected_features)
    test_df["v46_score"] = score_v46(test_df, trained, selected_features).astype(np.float32)

    gain = np.asarray(trained["model"].feature_importance(importance_type="gain"), dtype=float)
    split = np.asarray(trained["model"].feature_importance(importance_type="split"), dtype=float)
    if float(gain.sum()) > 0:
        gain_weights = gain / gain.sum()
    else:
        gain_weights = np.ones(len(gain), dtype=float) / float(len(gain))
    gain_weights = 0.95 * gain_weights + 0.05 / float(len(gain_weights))
    gain_weights = gain_weights / gain_weights.sum()
    for i, feature in enumerate(selected_features):
        feature_importance_rows.append({
            "fold_id": fold["fold_id"], "feature": feature,
            "importance_gain": float(gain[i]), "importance_split": float(split[i]),
            "distance_weight": float(gain_weights[i]),
        })

    # Strict historical pool: the reference label must be realized by train_end.
    hist_train_df = make_train_df(df_all, fold, label_safe=HISTORICAL_KNN_LABEL_SAFE)
    hist_pool_df = sample_history_pool(hist_train_df, HIST_POOL_PER_MONTH, SEED + fold_index * 101)
    hist_outputs, hist_distance = historical_knn_predictions(
        test_df, hist_pool_df, selected_rank_cols, HIST_K_LIST, weights=None,
    )
    for k in HIST_K_LIST:
        test_df["hist_knn%s" % k] = hist_outputs[k]
    test_df["hist_knn_distance"] = hist_distance

    month_groups = list(test_df.groupby(DATE_COL))
    for month_index, (dt, month_df_raw) in enumerate(progress_iter(month_groups, total=len(month_groups), desc="fold monthly collision", leave=False)):
        month_df = month_df_raw.sort_values(STOCK_COL).reset_index(drop=True)
        if len(month_df) < 100:
            continue
        rng = np.random.RandomState(SEED + fold_index * 10007 + month_index * 97)
        rank_x = fill_rank_matrix(month_df, selected_rank_cols)

        dist_equal = pairwise_rank_distance(rank_x, None)
        row, cases, nearest_equal = analyze_distance_matrix(month_df, dist_equal, "selected_equal", fold["fold_id"], rng)
        collision_monthly_rows.append(row)
        collision_case_rows.extend(cases)

        random_idx = random_industry_partner(month_df[INDUSTRY_COL].astype(str).values, rng)
        labels = np.asarray(month_df[TARGET_COL], dtype=float)
        true_rank = np.asarray(pd.Series(labels).rank(method="first", ascending=False), dtype=float)
        random_row, _ = pair_metrics(labels, random_idx, dist_equal[np.arange(len(month_df)), random_idx], true_rank, None)
        random_row.update({"fold_id": fold["fold_id"], DATE_COL: pd.Timestamp(dt), "method": "industry_random", "mutual_nearest_rate": np.nan})
        collision_monthly_rows.append(random_row)

        dist_gain = pairwise_rank_distance(rank_x, gain_weights)
        row, cases, nearest_gain = analyze_distance_matrix(month_df, dist_gain, "selected_gain", fold["fold_id"], rng)
        collision_monthly_rows.append(row)
        collision_case_rows.extend(cases)

        dist_score = score_distance_matrix(month_df["v46_score"])
        row, cases, nearest_score = analyze_distance_matrix(month_df, dist_score, "model_score", fold["fold_id"], rng)
        collision_monthly_rows.append(row)
        collision_case_rows.extend(cases)

        # Top30 candidate ambiguity under the actual V46 score.
        top_n = int(_bi.min(TOP_CANDIDATE_N, len(month_df)))
        top_df = month_df.sort_values("v46_score", ascending=False).head(top_n).reset_index(drop=True)
        top_x = fill_rank_matrix(top_df, selected_rank_cols)
        top_dist = pairwise_rank_distance(top_x, gain_weights)
        top_labels = np.asarray(top_df[TARGET_COL], dtype=float)
        top_true_rank = np.asarray(pd.Series(top_labels).rank(method="first", ascending=False), dtype=float)
        top_nearest = np.argmin(top_dist, axis=1)
        top_local_k = int(_bi.min(LOCAL_K, len(top_df) - 1))
        top_local = np.argpartition(top_dist, kth=top_local_k - 1, axis=1)[:, :top_local_k]
        top_row, _ = pair_metrics(top_labels, top_nearest, top_dist[np.arange(len(top_df)), top_nearest], top_true_rank, top_local)
        top_random = random_industry_partner(top_df[INDUSTRY_COL].astype(str).values, rng)
        top_random_row, _ = pair_metrics(top_labels, top_random, top_dist[np.arange(len(top_df)), top_random], top_true_rank, None)
        top_row.update({
            "fold_id": fold["fold_id"], DATE_COL: pd.Timestamp(dt), "candidate_count": int(len(top_df)),
            "random_abs_label_gap_mean": top_random_row["abs_label_gap_mean"],
            "random_sign_conflict_rate": top_random_row["sign_conflict_rate"],
            "gap_reduction_vs_random": 1.0 - top_row["abs_label_gap_mean"] / top_random_row["abs_label_gap_mean"] if top_random_row["abs_label_gap_mean"] > 0 else np.nan,
        })
        top30_collision_rows.append(top_row)

        # Factor-group geometry on a deterministic monthly subsample.
        sample_n = int(_bi.min(GROUP_SAMPLE_PER_MONTH, len(month_df)))
        sample_idx = rng.choice(np.arange(len(month_df)), size=sample_n, replace=False)
        sample_df = month_df.iloc[sample_idx].reset_index(drop=True)
        sample_labels = np.asarray(sample_df[TARGET_COL], dtype=float)
        sample_true_rank = np.asarray(pd.Series(sample_labels).rank(method="first", ascending=False), dtype=float)
        sample_random = random_industry_partner(sample_df[INDUSTRY_COL].astype(str).values, rng)
        for group_name, group_features in FACTOR_GROUPS.items():
            group_rank_cols = [RANK_PREFIX + c for c in group_features if c in FULL_FEATURE_COLS]
            if len(group_rank_cols) == 0:
                continue
            group_x = fill_rank_matrix(sample_df, group_rank_cols)
            group_dist = pairwise_rank_distance(group_x, None)
            group_nearest = np.argmin(group_dist, axis=1)
            group_metrics, _ = pair_metrics(sample_labels, group_nearest, group_dist[np.arange(sample_n), group_nearest], sample_true_rank, None)
            group_random_metrics, _ = pair_metrics(sample_labels, sample_random, group_dist[np.arange(sample_n), sample_random], sample_true_rank, None)
            group_collision_rows.append({
                "fold_id": fold["fold_id"], DATE_COL: pd.Timestamp(dt), "factor_group": group_name,
                "feature_count": int(len(group_rank_cols)), "sample_count": sample_n,
                "neighbor_abs_label_gap_mean": group_metrics["abs_label_gap_mean"],
                "random_abs_label_gap_mean": group_random_metrics["abs_label_gap_mean"],
                "gap_reduction_vs_random": 1.0 - group_metrics["abs_label_gap_mean"] / group_random_metrics["abs_label_gap_mean"] if group_random_metrics["abs_label_gap_mean"] > 0 else np.nan,
                "neighbor_sign_conflict_rate": group_metrics["sign_conflict_rate"],
                "random_sign_conflict_rate": group_random_metrics["sign_conflict_rate"],
                "neighbor_top20_bottom20_collision_rate": group_metrics["top20_bottom20_collision_rate"],
            })
            del group_x, group_dist

        # Strict OOS historical KNN versus V46 score.
        for method in ["v46_score"] + ["hist_knn%s" % k for k in HIST_K_LIST]:
            metrics = top8_metrics(month_df[TARGET_COL], month_df[method])
            metrics.update({
                "fold_id": fold["fold_id"], DATE_COL: pd.Timestamp(dt), "method": method,
                "n": int(len(month_df)), "historical_pool_rows": int(len(hist_pool_df)),
                "historical_pool_months": int(hist_pool_df[DATE_COL].nunique()),
                "historical_pool_date_max": hist_pool_df[DATE_COL].max(),
                "mean_knn_distance": float(month_df["hist_knn_distance"].mean()) if method.startswith("hist_knn") else np.nan,
            })
            historical_knn_rows.append(metrics)

        del rank_x, dist_equal, dist_gain, dist_score, top_x, top_dist
        gc.collect()

    model_meta_rows.append({
        "fold_id": fold["fold_id"], "train_start": fold["train_start"], "train_end": fold["train_end"],
        "test_start": fold["test_start"], "test_end": fold["test_end"],
        "train_rows": int(len(train_df)), "train_months": int(train_df[DATE_COL].nunique()),
        "hist_label_safe_rows": int(len(hist_train_df)), "hist_pool_rows": int(len(hist_pool_df)),
        "hist_pool_months": int(hist_pool_df[DATE_COL].nunique()), "test_rows": int(len(test_df)),
        "test_months": int(test_df[DATE_COL].nunique()), "feature_count": int(len(selected_features)),
        "removed_feature_count": int(len(removed_features)), "train_rank_ic": trained["train_rank_ic"],
        "selected_features": ",".join(selected_features), "removed_features": ",".join(removed_features),
    })
    del trained, train_df, test_df, hist_train_df, hist_pool_df, hist_outputs, hist_distance
    gc.collect()

collision_monthly_df = pd.DataFrame(collision_monthly_rows)
collision_cases_df = pd.DataFrame(collision_case_rows)
if len(collision_cases_df):
    collision_cases_df["pair_key"] = collision_cases_df.apply(lambda r: "|".join(_bi.sorted([str(r["stock"]), str(r["neighbor_stock"])])), axis=1)
    collision_cases_df = collision_cases_df.sort_values("abs_label_gap", ascending=False).drop_duplicates([DATE_COL, "method", "pair_key"], keep="first")
top30_collision_df = pd.DataFrame(top30_collision_rows)
group_collision_df = pd.DataFrame(group_collision_rows)
historical_knn_monthly_df = pd.DataFrame(historical_knn_rows)
model_meta_df = pd.DataFrame(model_meta_rows)
feature_importance_df = pd.DataFrame(feature_importance_rows)

collision_monthly_df.to_csv(OUT_DIR / "v86_collision_monthly.csv", index=False)
collision_cases_df.to_csv(OUT_DIR / "v86_collision_cases.csv", index=False)
top30_collision_df.to_csv(OUT_DIR / "v86_top30_collision_monthly.csv", index=False)
group_collision_df.to_csv(OUT_DIR / "v86_factor_group_collision_monthly.csv", index=False)
historical_knn_monthly_df.to_csv(OUT_DIR / "v86_historical_knn_monthly.csv", index=False)
model_meta_df.to_csv(OUT_DIR / "v86_model_meta.csv", index=False)
feature_importance_df.to_csv(OUT_DIR / "v86_feature_importance.csv", index=False)
display_df(model_meta_df, 20)
display_df(collision_monthly_df, 20)


## 7. 汇总、年度稳定性与预注册判定


In [ ]:
collision_metric_cols = [
    "abs_label_gap_mean", "abs_label_gap_median", "sign_conflict_rate",
    "label_gap_gt10_rate", "label_gap_gt20_rate", "top20_bottom20_collision_rate",
    "local_std_ratio", "same_month_knn_rank_ic", "mutual_nearest_rate",
]
collision_summary_rows = []
for method, gdf in collision_monthly_df.groupby("method"):
    row = {"method": method, "months": int(len(gdf))}
    for col in collision_metric_cols:
        row[col + "_mean"] = float(pd.to_numeric(gdf[col], errors="coerce").mean())
        row[col + "_median"] = float(pd.to_numeric(gdf[col], errors="coerce").median())
    collision_summary_rows.append(row)
collision_summary_df = pd.DataFrame(collision_summary_rows)

random_summary = collision_summary_df[collision_summary_df["method"] == "industry_random"]
random_gap = float(random_summary["abs_label_gap_mean_mean"].iloc[0]) if len(random_summary) else np.nan
random_sign = float(random_summary["sign_conflict_rate_mean"].iloc[0]) if len(random_summary) else np.nan
random_extreme = float(random_summary["top20_bottom20_collision_rate_mean"].iloc[0]) if len(random_summary) else np.nan
collision_summary_df["label_gap_reduction_vs_random"] = 1.0 - collision_summary_df["abs_label_gap_mean_mean"] / random_gap if random_gap > 0 else np.nan
collision_summary_df["sign_conflict_reduction_vs_random"] = random_sign - collision_summary_df["sign_conflict_rate_mean"] if np.isfinite(random_sign) else np.nan
collision_summary_df["extreme_collision_reduction_vs_random"] = 1.0 - collision_summary_df["top20_bottom20_collision_rate_mean"] / random_extreme if random_extreme > 0 else np.nan

collision_year_rows = []
collision_monthly_df["year"] = pd.to_datetime(collision_monthly_df[DATE_COL]).dt.year
for (method, year), gdf in collision_monthly_df.groupby(["method", "year"]):
    row = {"method": method, "year": int(year), "months": int(len(gdf))}
    for col in collision_metric_cols:
        row[col + "_mean"] = float(pd.to_numeric(gdf[col], errors="coerce").mean())
    collision_year_rows.append(row)
collision_yearly_df = pd.DataFrame(collision_year_rows)
random_year = collision_yearly_df[collision_yearly_df["method"] == "industry_random"][["year", "abs_label_gap_mean_mean"]].rename(columns={"abs_label_gap_mean_mean": "random_gap"})
collision_yearly_df = pd.merge(collision_yearly_df, random_year, on="year", how="left")
collision_yearly_df["label_gap_reduction_vs_random"] = 1.0 - collision_yearly_df["abs_label_gap_mean_mean"] / collision_yearly_df["random_gap"]

group_summary_rows = []
for group_name, gdf in group_collision_df.groupby("factor_group"):
    group_summary_rows.append({
        "factor_group": group_name, "months": int(len(gdf)),
        "feature_count": int(gdf["feature_count"].max()),
        "neighbor_abs_label_gap_mean": float(gdf["neighbor_abs_label_gap_mean"].mean()),
        "random_abs_label_gap_mean": float(gdf["random_abs_label_gap_mean"].mean()),
        "gap_reduction_vs_random": float(gdf["gap_reduction_vs_random"].mean()),
        "neighbor_sign_conflict_rate": float(gdf["neighbor_sign_conflict_rate"].mean()),
        "random_sign_conflict_rate": float(gdf["random_sign_conflict_rate"].mean()),
        "neighbor_top20_bottom20_collision_rate": float(gdf["neighbor_top20_bottom20_collision_rate"].mean()),
    })
group_collision_summary_df = pd.DataFrame(group_summary_rows)

top30_summary_df = pd.DataFrame([{
    "months": int(len(top30_collision_df)),
    "candidate_count_mean": float(top30_collision_df["candidate_count"].mean()),
    "neighbor_abs_label_gap_mean": float(top30_collision_df["abs_label_gap_mean"].mean()),
    "random_abs_label_gap_mean": float(top30_collision_df["random_abs_label_gap_mean"].mean()),
    "gap_reduction_vs_random": float(top30_collision_df["gap_reduction_vs_random"].mean()),
    "neighbor_sign_conflict_rate": float(top30_collision_df["sign_conflict_rate"].mean()),
    "random_sign_conflict_rate": float(top30_collision_df["random_sign_conflict_rate"].mean()),
    "local_std_ratio": float(top30_collision_df["local_std_ratio"].mean()),
    "same_month_knn_rank_ic": float(top30_collision_df["same_month_knn_rank_ic"].mean()),
}])

historical_summary_rows = []
for method, gdf in historical_knn_monthly_df.groupby("method"):
    historical_summary_rows.append({
        "method": method, "months": int(len(gdf)),
        "rank_ic_mean": float(gdf["rank_ic"].mean()), "rank_ic_median": float(gdf["rank_ic"].median()),
        "rank_ic_positive_rate": float((gdf["rank_ic"] > 0).mean()),
        "precision_at8_true20_mean": float(gdf["precision_at8_true20"].mean()),
        "recall_at8_true20_mean": float(gdf["recall_at8_true20"].mean()),
        "top8_edge_mean": float(gdf["top8_edge"].mean()),
        "top_decile_alpha_mean": float(gdf["top_decile_alpha"].mean()),
        "mean_knn_distance": float(gdf["mean_knn_distance"].mean()),
    })
historical_knn_summary_df = pd.DataFrame(historical_summary_rows)

historical_knn_monthly_df["year"] = pd.to_datetime(historical_knn_monthly_df[DATE_COL]).dt.year
historical_year_rows = []
for (method, year), gdf in historical_knn_monthly_df.groupby(["method", "year"]):
    historical_year_rows.append({
        "method": method, "year": int(year), "months": int(len(gdf)),
        "rank_ic_mean": float(gdf["rank_ic"].mean()),
        "precision_at8_true20_mean": float(gdf["precision_at8_true20"].mean()),
        "top8_edge_mean": float(gdf["top8_edge"].mean()),
        "top_decile_alpha_mean": float(gdf["top_decile_alpha"].mean()),
    })
historical_knn_yearly_df = pd.DataFrame(historical_year_rows)

selected_row = collision_summary_df[collision_summary_df["method"] == "selected_equal"]
hist_row = historical_knn_summary_df[historical_knn_summary_df["method"] == "hist_knn5"]
if len(selected_row) == 0 or len(hist_row) == 0:
    raise ValueError("missing selected_equal or hist_knn5 summary")
selected_row = selected_row.iloc[0]
hist_row = hist_row.iloc[0]
hist_year = historical_knn_yearly_df[historical_knn_yearly_df["method"] == "hist_knn5"]
positive_hist_years = int((hist_year["rank_ic_mean"] > 0).sum())

structural_gates = {
    "no_duplicate_stock_date": duplicate_stock_date == 0,
    "no_nonfinite_target": nonfinite_target == 0,
    "no_exact_feature_conflict_gt10pct": exact_duplicate_conflict_groups == 0,
}
separability_gates = {
    "gap_reduction_ge_25pct": float(selected_row["label_gap_reduction_vs_random"]) >= GOOD_GAP_REDUCTION,
    "sign_conflict_le_35pct": float(selected_row["sign_conflict_rate_mean"]) <= GOOD_SIGN_CONFLICT_MAX,
    "local_std_ratio_le_75pct": float(selected_row["local_std_ratio_mean"]) <= GOOD_LOCAL_STD_RATIO_MAX,
    "historical_knn_rank_ic_ge_003": float(hist_row["rank_ic_mean"]) >= GOOD_HIST_KNN_RANK_IC,
    "historical_knn_positive_years_ge_3": positive_hist_years >= 3,
}
if _bi.all(structural_gates.values()):
    structural_quality = "pass"
else:
    structural_quality = "review_data_pipeline"
if _bi.all(separability_gates.values()):
    conditional_separability = "high"
elif float(selected_row["label_gap_reduction_vs_random"]) >= MIXED_GAP_REDUCTION or float(hist_row["rank_ic_mean"]) >= MIXED_HIST_KNN_RANK_IC:
    conditional_separability = "mixed_partial_signal"
elif float(selected_row["sign_conflict_rate_mean"]) >= BAD_SIGN_CONFLICT_MIN and float(selected_row["local_std_ratio_mean"]) >= BAD_LOCAL_STD_RATIO_MIN:
    conditional_separability = "high_conditional_noise"
else:
    conditional_separability = "low_uncertain"

decision = "model_improvement_still_promising" if conditional_separability == "high" else (
    "targeted_feature_work" if conditional_separability == "mixed_partial_signal" else "feature_information_bottleneck"
)
decision_row = {
    "structural_quality": structural_quality,
    "conditional_separability": conditional_separability,
    "research_decision": decision,
    "selected_neighbor_gap_reduction_vs_random": float(selected_row["label_gap_reduction_vs_random"]),
    "selected_neighbor_sign_conflict_rate": float(selected_row["sign_conflict_rate_mean"]),
    "selected_neighbor_local_std_ratio": float(selected_row["local_std_ratio_mean"]),
    "same_month_knn_rank_ic": float(selected_row["same_month_knn_rank_ic_mean"]),
    "historical_knn5_rank_ic": float(hist_row["rank_ic_mean"]),
    "historical_knn5_positive_years": positive_hist_years,
    "top30_gap_reduction_vs_random": float(top30_summary_df["gap_reduction_vs_random"].iloc[0]),
    "exact_duplicate_conflict_groups": int(exact_duplicate_conflict_groups),
}
decision_row.update(structural_gates)
decision_row.update(separability_gates)
decision_df = pd.DataFrame([decision_row])

collision_summary_df.to_csv(OUT_DIR / "v86_collision_summary.csv", index=False)
collision_yearly_df.to_csv(OUT_DIR / "v86_collision_yearly.csv", index=False)
group_collision_summary_df.to_csv(OUT_DIR / "v86_factor_group_collision_summary.csv", index=False)
top30_summary_df.to_csv(OUT_DIR / "v86_top30_collision_summary.csv", index=False)
historical_knn_summary_df.to_csv(OUT_DIR / "v86_historical_knn_summary.csv", index=False)
historical_knn_yearly_df.to_csv(OUT_DIR / "v86_historical_knn_yearly.csv", index=False)
decision_df.to_csv(OUT_DIR / "v86_pre_registered_decision.csv", index=False)

display_df(collision_summary_df, 20)
display_df(group_collision_summary_df.sort_values("gap_reduction_vs_random", ascending=False), 20)
display_df(top30_summary_df, 10)
display_df(historical_knn_summary_df, 20)
display_df(decision_df, 10)


## 8. 可视化仪表板


In [ ]:
# 1) Label tails and feature quality.
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes[0, 0].hist(np.asarray(df_all[TARGET_COL].dropna(), dtype=float), bins=100, color="#2f5597", alpha=0.85)
axes[0, 0].set_xlim(float(df_all[TARGET_COL].quantile(0.005)), float(df_all[TARGET_COL].quantile(0.995)))
axes[0, 0].set_title("alpha_1m distribution (0.5%-99.5%)")
yearly_tail = label_monthly_df.groupby("year")[["std", "top5pct_centered_l2_share"]].mean()
axes[0, 1].plot(yearly_tail.index, yearly_tail["std"], marker="o", color="#e64b35", label="monthly std")
axes[0, 1].plot(yearly_tail.index, yearly_tail["top5pct_centered_l2_share"], marker="o", color="#00a087", label="top5% L2 share")
axes[0, 1].set_title("Label noise and tail concentration by year")
axes[0, 1].legend(fontsize=8)
missing_plot = feature_quality_df.sort_values("missing_rate", ascending=False).head(15)
axes[1, 0].barh(np.arange(len(missing_plot)), missing_plot["missing_rate"].values, color="#f0a202")
axes[1, 0].set_yticks(np.arange(len(missing_plot)))
axes[1, 0].set_yticklabels(missing_plot["feature"].values, fontsize=8)
axes[1, 0].invert_yaxis()
axes[1, 0].set_title("Highest feature missing rates")
stale_plot = feature_quality_df.sort_values("consecutive_same_value_rate", ascending=False).head(15)
axes[1, 1].barh(np.arange(len(stale_plot)), stale_plot["consecutive_same_value_rate"].values, color="#7e57c2")
axes[1, 1].set_yticks(np.arange(len(stale_plot)))
axes[1, 1].set_yticklabels(stale_plot["feature"].values, fontsize=8)
axes[1, 1].invert_yaxis()
axes[1, 1].set_title("Highest consecutive unchanged rates")
for ax in axes.ravel():
    ax.grid(alpha=0.25)
save_show(fig, "v86_data_quality_and_label_tail.png")

# 2) Nearest-neighbor versus random collision dashboard.
methods = [m for m in ["selected_equal", "selected_gain", "model_score", "industry_random"] if m in set(collision_summary_df["method"])]
metric_specs = [
    ("abs_label_gap_mean_mean", "Mean absolute label gap"),
    ("sign_conflict_rate_mean", "Sign-conflict rate"),
    ("label_gap_gt10_rate_mean", "Label gap > 10%"),
    ("label_gap_gt20_rate_mean", "Label gap > 20%"),
    ("top20_bottom20_collision_rate_mean", "Top20-Bottom20 collision"),
    ("local_std_ratio_mean", "Local / universe label std"),
    ("same_month_knn_rank_ic_mean", "Same-month KNN RankIC"),
    ("label_gap_reduction_vs_random", "Gap reduction vs random"),
]
fig, axes = plt.subplots(2, 4, figsize=(19, 9))
for ax, (col, title) in zip(axes.ravel(), metric_specs):
    values = []
    for method in methods:
        values.append(float(collision_summary_df.loc[collision_summary_df["method"] == method, col].iloc[0]))
    ax.bar(np.arange(len(methods)), values, color=[COLORS.get(m, "#777777") for m in methods])
    ax.set_xticks(np.arange(len(methods)))
    ax.set_xticklabels([m.replace("selected_", "") for m in methods], rotation=25, ha="right", fontsize=8)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
save_show(fig, "v86_collision_metric_dashboard.png")

# 3) Yearly collision stability.
equal_year = collision_yearly_df[collision_yearly_df["method"] == "selected_equal"].sort_values("year")
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].bar(equal_year["year"].astype(str), equal_year["label_gap_reduction_vs_random"], color="#2f5597")
axes[0].axhline(0, color="#555555", linewidth=1)
axes[0].set_title("Label-gap reduction vs random")
axes[1].bar(equal_year["year"].astype(str), equal_year["sign_conflict_rate_mean"], color="#e64b35")
axes[1].axhline(0.5, color="#555555", linestyle="--")
axes[1].set_title("Nearest-neighbor sign conflict")
axes[2].bar(equal_year["year"].astype(str), equal_year["local_std_ratio_mean"], color="#00a087")
axes[2].axhline(1.0, color="#555555", linestyle="--")
axes[2].set_title("Local / universe label std")
for ax in axes:
    ax.grid(axis="y", alpha=0.25)
save_show(fig, "v86_collision_yearly_stability.png")

# 4) Factor-group separability.
group_plot = group_collision_summary_df.sort_values("gap_reduction_vs_random", ascending=True)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
y = np.arange(len(group_plot))
axes[0].barh(y, group_plot["gap_reduction_vs_random"].values, color="#2f5597")
axes[0].set_yticks(y)
axes[0].set_yticklabels(group_plot["factor_group"].values)
axes[0].axvline(0, color="#555555", linewidth=1)
axes[0].set_title("Label-gap reduction by factor group")
axes[1].barh(y, group_plot["neighbor_sign_conflict_rate"].values, color="#e64b35")
axes[1].set_yticks(y)
axes[1].set_yticklabels(group_plot["factor_group"].values)
axes[1].axvline(0.5, color="#555555", linestyle="--")
axes[1].set_title("Sign conflict by factor group")
for ax in axes:
    ax.grid(alpha=0.25)
save_show(fig, "v86_factor_group_separability.png")

# 5) Strict historical OOS KNN compared with V46.
hist_methods = [m for m in ["v46_score", "hist_knn5", "hist_knn20"] if m in set(historical_knn_summary_df["method"])]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, title in [
    (axes[0], "rank_ic_mean", "OOS RankIC"),
    (axes[1], "precision_at8_true20_mean", "OOS Precision@8: true Top20"),
    (axes[2], "top8_edge_mean", "OOS Top8 edge"),
]:
    vals = [float(historical_knn_summary_df.loc[historical_knn_summary_df["method"] == m, col].iloc[0]) for m in hist_methods]
    ax.bar(np.arange(len(hist_methods)), vals, color=[COLORS.get(m, "#777777") for m in hist_methods])
    ax.set_xticks(np.arange(len(hist_methods)))
    ax.set_xticklabels(hist_methods, rotation=20, ha="right")
    ax.axhline(0, color="#555555", linewidth=1)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
save_show(fig, "v86_historical_knn_dashboard.png")

fig, ax = plt.subplots(figsize=(12, 5.5))
for method in hist_methods:
    d = historical_knn_yearly_df[historical_knn_yearly_df["method"] == method].sort_values("year")
    ax.plot(d["year"], d["rank_ic_mean"], marker="o", color=COLORS.get(method), label=method)
ax.axhline(0, color="#555555", linewidth=1)
ax.set_title("Historical KNN OOS RankIC by year")
ax.set_ylabel("RankIC")
ax.grid(alpha=0.25)
ax.legend()
save_show(fig, "v86_historical_knn_yearly.png")

# 6) Top30 ambiguity versus the full-universe selected-feature geometry.
full_equal = collision_summary_df[collision_summary_df["method"] == "selected_equal"].iloc[0]
compare_labels = ["full universe", "V46 Top30"]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].bar(compare_labels, [full_equal["abs_label_gap_mean_mean"], top30_summary_df["neighbor_abs_label_gap_mean"].iloc[0]], color=["#2f5597", "#f0a202"])
axes[0].set_title("Nearest-neighbor label gap")
axes[1].bar(compare_labels, [full_equal["sign_conflict_rate_mean"], top30_summary_df["neighbor_sign_conflict_rate"].iloc[0]], color=["#2f5597", "#f0a202"])
axes[1].set_title("Sign-conflict rate")
axes[2].bar(compare_labels, [full_equal["local_std_ratio_mean"], top30_summary_df["local_std_ratio"].iloc[0]], color=["#2f5597", "#f0a202"])
axes[2].set_title("Local / universe label std")
for ax in axes:
    ax.grid(axis="y", alpha=0.25)
save_show(fig, "v86_top30_ambiguity.png")


## 9. 输出文件与解读边界


In [ ]:
readme = [
    "V86 sample collision audit interpretation notes",
    "1. Same-month neighbors use realized labels only to measure ex-post feature separability; they are not a tradable prediction.",
    "2. Historical KNN is strict OOS: reference labels must be realized by each fold cutoff, and the same stock is excluded from its neighbor pool.",
    "3. Monthly cross-sectional rank normalization makes factor scales comparable and avoids future information.",
    "4. selected_equal is the primary geometry; selected_gain tests the geometry emphasized by the exact V46 model.",
    "5. industry_random is the matched baseline. Interpret reductions versus this baseline, not raw gap levels alone.",
    "6. Exact duplicates and nonfinite labels diagnose pipeline cleanliness. Near-neighbor disagreement diagnoses conditional label noise or omitted information.",
    "7. Fundamental factors can legitimately remain unchanged across months; stale-value rates are diagnostics, not automatic errors.",
    "8. Top30 diagnostics are especially relevant to an eight-stock strategy because ambiguity near the selection boundary drives holding instability.",
    "9. High-dimensional nearest-neighbor results approximate, but do not identify, the theoretical Bayes error.",
    "10. The decision table selects the next research direction; it does not approve a live strategy.",
]
with open(OUT_DIR / "v86_README.txt", "w") as f:
    f.write("\n".join(readme))

print("saved outputs:")
for path in _bi.sorted(OUT_DIR.glob("v86_*")):
    print("-", path)
print("figures:")
for path in _bi.sorted(FIG_DIR.glob("*.png")):
    print("-", path)
print("\nPrimary result:")
display_df(decision_df, 10)
